# grad-accumulate-on-leaf — faded example 2: Faded: implement zero_grad to reset leaf gradients to None

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-accumulate-on-leaf`. Running the beacon reports progress on the `Backprop: Grad accumulate on leaf` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad accumulate on leaf` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-accumulate-on-leaf`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-accumulate-on-leaf"
DD_SUBTOPIC = "Backprop: Grad accumulate on leaf"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Between training steps, `zero_grad` clears accumulated gradients so that each step's backward pass starts with a fresh accumulation. PyTorch 2.0+ defaults to setting `.grad = None` rather than zeroing a buffer, because `None` causes the next `accumulate_grad` first-touch to avoid an unnecessary zero-tensor allocation. Pass through any iterable of parameters — not just lists.

## Faded exercise 2

The `accumulate_grad` function is already correct. Complete `zero_grad` so that it sets each parameter's `.grad` to `None`.

The blank is the single statement inside the loop body that clears the gradient.

**Fill in:** The statement that sets p.grad to None, clearing the accumulated gradient for this parameter.

In [ ]:
import torch as t

t.manual_seed(0)

class SimpleLeaf:
    def __init__(self):
        self.grad = None

def accumulate_grad(leaf, g):
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g

def zero_grad(params):
    for p in params:
        raise NotImplementedError()  # TODO: The statement that sets p.grad to None, clearing the accumulated gradient for this parameter.

# Exercise it
p1 = SimpleLeaf()
p2 = SimpleLeaf()
accumulate_grad(p1, t.tensor([1.0, 2.0]))
accumulate_grad(p2, t.tensor([3.0, 4.0]))
zero_grad([p1, p2])
print(p1.grad, p2.grad)  # should be None None


def _test():
    import torch as t

    class SimpleLeaf:
        def __init__(self):
            self.grad = None

    params = [SimpleLeaf() for _ in range(4)]
    gs = [t.ones(3) * float(i) for i in range(4)]
    for p, g in zip(params, gs):
        accumulate_grad(p, g)

    # All should have grads before zero_grad
    assert all(p.grad is not None for p in params)

    zero_grad(params)
    assert all(p.grad is None for p in params), "zero_grad must set all .grad to None"

    # After zero_grad, the first accumulate should land cleanly as first-touch
    accumulate_grad(params[0], t.tensor([9.0, 9.0, 9.0]))
    assert t.allclose(params[0].grad, t.tensor([9.0, 9.0, 9.0])), \
        "Post-zero first-touch should equal the new gradient exactly"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class SimpleLeaf:
    def __init__(self):
        self.grad = None

def accumulate_grad(leaf, g):
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g

def zero_grad(params):
    for p in params:
        p.grad = None

# Exercise it
p1 = SimpleLeaf()
p2 = SimpleLeaf()
accumulate_grad(p1, t.tensor([1.0, 2.0]))
accumulate_grad(p2, t.tensor([3.0, 4.0]))
zero_grad([p1, p2])
print(p1.grad, p2.grad)  # should be None None
```
</details>